In [575]:
import requests
import re
from PIL import Image
from io import BytesIO
import ezdxf
from shapely.geometry import shape

from __future__ import annotations

import math
from typing import Dict, List, Tuple, Optional, Any

from PIL import Image
import xml.etree.ElementTree as ET
import numpy as np

from rasterio.io import MemoryFile

import os

In [576]:
adres = "Julianaweg 22 7078 AR Megchelen"
grootte = 250

In [577]:
SUGGEST = "https://api.pdok.nl/bzk/locatieserver/search/v3_1/suggest"
LOOKUP  = "https://api.pdok.nl/bzk/locatieserver/search/v3_1/lookup"
KADAS = "https://api.pdok.nl/kadaster/brk-kadastrale-kaart/ogc/v1"

RD_CRS = "http://www.opengis.net/def/crs/EPSG/0/28992"

Gebruikt API van de locatieserver om adres om te zetten naar RD coördinaten.

In [578]:
def adres_naar_rd(adres):
    # 1) Suggest: zoek beste match + id
    r = requests.get(SUGGEST, params={"q": adres}, timeout=30)
    r.raise_for_status()
    docs = r.json()["response"]["docs"]
    if not docs:
        raise ValueError("Geen resultaat voor dit adres.")
    loc_id = docs[0]["id"]

    # 2) Lookup: haal details van die id
    r = requests.get(LOOKUP, params={"id": loc_id}, timeout=30)
    r.raise_for_status()
    doc = r.json()["response"]["docs"][0]

    # 3) Haal RD-coördinaten op
    rd = doc["centroide_rd"]
    cleaned_rd = re.sub(r"[^0-9. ]", "", rd)
    x, y = map(float, cleaned_rd.split())
    

    return x, y

In [579]:
x, y = adres_naar_rd(adres)
meters = grootte #Grootte van de bbox in meters
bbox = (x-meters, y-meters, x+meters, y+meters)
bbox_klein = (x-0.1, y-0.1, x+0.1, y+0.1)

In [580]:
def perceel_informatie(bbox_klein):  
    bbox_klein_str = ",".join(map(str, bbox_klein))

    r = requests.get(f"{KADAS}/collections/perceel/items",params={"bbox": bbox_klein_str, "bbox-crs": RD_CRS},
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    
    percelen = []
    for feature in data["features"]:
        props = feature["properties"]

        percelen.append({
            "perceelnummer": props.get("perceelnummer"),
            "sectie": props.get("sectie"),
            "oppervlakte_m2": props.get("kadastrale_grootte_waarde"),
            "definitief?": props.get("soort_grootte_waarde"),
            "status": props.get("status_historie_waarde")
        })
    return percelen

In [581]:
print(perceel_informatie(bbox_klein))

[{'perceelnummer': 1057, 'sectie': 'I', 'oppervlakte_m2': 1774, 'definitief?': 'Vastgesteld', 'status': 'Geldig'}]


In [582]:
def bebouwing_informatie(bbox_klein):  
    bbox_klein_str = ",".join(map(str, bbox_klein))

    r = requests.get(f"{KADAS}/collections/bebouwing/items",params={"bbox": bbox_klein_str, "bbox-crs": RD_CRS},
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    
    bebouwing = []
    for feature in data["features"]:
        props = feature["properties"]

        bebouwing.append({
            "relatieve hoogteligging": props.get("relatieve_hoogteligging")
        })
    return bebouwing

In [583]:
print(bebouwing_informatie(bbox_klein))

[{'relatieve hoogteligging': 0}]


In [584]:
WMS_URL = "https://service.pdok.nl/hwh/luchtfotorgb/wms/v1_0"
PLU_WMS = "https://service.pdok.nl/kadaster/plu/wms/v1_0"
KAD_WMS = "https://service.pdok.nl/kadaster/kadastralekaart/wms/v5_0?"

In [585]:
def wms_to_image(wms_url, params):
    r = requests.get(wms_url, params=params, timeout=60)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGBA")



In [586]:
def wms_parameters(bbox, width=2000, height=2000):
    return {
        "service": "WMS",
        "request": "GetMap",
        "version": "1.3.0",
        "crs": "EPSG:28992",
        "bbox": ",".join(map(str, bbox)),
        "width": str(width),
        "height": str(height),
        "format": "image/png",
        "transparent": "true",
    }

In [587]:
params = wms_parameters(bbox)

# 1) luchtfoto
luchtfoto = wms_to_image(WMS_URL, {
    **params,
    "layers": "Actueel_orthoHR",
    "styles": "",
})

# 2) bestemmingsplan
bestemming = wms_to_image(PLU_WMS, {
    **params,
    "layers": "enkelbestemming",
    "styles": "enkelbestemming",
})

# 3) perceelgrenzen
percelen = wms_to_image(KAD_WMS, {
    **params,
    "layers": "kadastralekaart",
    "styles": "",
})

In [588]:
def place_legend_on_image(
    base: Image.Image,
    legend: Image.Image,
    position: str = "bottom-right",
    margin: int = 30,
    legend_scale: float = 1.5,          # maak legenda eerst groter
    legend_max_width_ratio: float = 0.25,  # max 25% van kaartbreedte
    add_white_box: bool = True,
    box_padding: int = 14,
):
    base = base.convert("RGBA")
    legend = legend.convert("RGBA")

    # 1) eerst "grof" schalen zodat hij niet mini is
    legend = legend.resize(
        (int(legend.size[0] * legend_scale), int(legend.size[1] * legend_scale)),
        Image.Resampling.LANCZOS
    )

    # 2) daarna cap op max breedte t.o.v. kaart
    max_w = int(base.size[0] * legend_max_width_ratio)
    if legend.size[0] > max_w:
        scale = max_w / legend.size[0]
        legend = legend.resize(
            (max_w, int(legend.size[1] * scale)),
            Image.Resampling.LANCZOS
        )

    # 3) optioneel wit vlak erachter
    if add_white_box:
        box_w = legend.size[0] + 2 * box_padding
        box_h = legend.size[1] + 2 * box_padding
        box = Image.new("RGBA", (box_w, box_h), (255, 255, 255, 220))
        box.paste(legend, (box_padding, box_padding), legend)
        legend = box

    # 4) positie bepalen
    W, H = base.size
    w, h = legend.size
    if position == "bottom-right":
        x, y = W - w - margin, H - h - margin
    elif position == "bottom-left":
        x, y = margin, H - h - margin
    elif position == "top-right":
        x, y = W - w - margin, margin
    elif position == "top-left":
        x, y = margin, margin
    else:
        raise ValueError("position must be one of: bottom-right, bottom-left, top-right, top-left")

    out = base.copy()
    out.paste(legend, (x, y), legend)
    return out




In [589]:
def wms_legend_to_image(wms_url: str, layer: str, style: str = "", version: str = "1.3.0") -> Image.Image:
    params = {
        "service": "WMS",
        "request": "GetLegendGraphic",
        "version": version,
        "format": "image/png",
        "layer": layer,
    }
    if style:
        params["style"] = style

    r = requests.get(wms_url, params=params, timeout=60)
    r.raise_for_status()
    return Image.open(BytesIO(r.content))

In [590]:
legend_url = "https://service.pdok.nl/kadaster/plu/wms/v1_0/legend/enkelbestemming/enkelbestemming.png"
legend_img = Image.open(BytesIO(requests.get(legend_url, timeout=30).content)).convert("RGBA")

In [591]:
plu_plus_percelen = Image.alpha_composite(bestemming, percelen)

# legenda erop
bestemming_kadaster = place_legend_on_image(
    base=plu_plus_percelen,
    legend=legend_img,
    position="bottom-right",
    legend_scale=2.0,           
    legend_max_width_ratio=0.2
)

# alleen eindproduct opslaan
bestemming_kadaster.save("Bestemming_percelen.png")
luchtfoto_plus_percelen = Image.alpha_composite(luchtfoto, percelen)
luchtfoto_plus_percelen.save("luchtfoto_percelen.png")

CAD ONDERLEGGER

In [592]:
AHN_WMS = "https://service.pdok.nl/rws/ahn/wms/v1_0"

In [593]:
def download_ahn_png(bbox_rd, out_path="ahn_kaart.png", width=2000, height=2000, add_legend=True):
    params = wms_parameters(bbox_rd, width=width, height=height)

    layer = "dtm_05m"
    style = "default"

    ahn_img = wms_to_image(AHN_WMS, {
        **params,
        "layers": layer,
        "styles": style,
    })

    if add_legend:
        try:
            legend_url = (
                "https://service.pdok.nl/rws/actueel-hoogtebestand-nederland/wms/v1_0"
                "?language=dut&version=1.3.0&service=WMS&request=GetLegendGraphic"
                "&sld_version=1.1.0&layer=dtm_05m&format=image/png&STYLE=default"
            )

            r = requests.get(legend_url, timeout=60)
            r.raise_for_status()
            legend = Image.open(BytesIO(r.content))

            ahn_img = place_legend_on_image(ahn_img, legend, position="bottom-right")
        except Exception as e:
            print(f"[WARN] AHN legenda ophalen mislukt: {e}")

    ahn_img.save(out_path)
    return os.path.abspath(out_path)

In [594]:
WDM_WMS = "https://service.pdok.nl/bzk/bro-grondwaterspiegeldiepte/wms/v2_0"

In [595]:
def wdm_legend_image(layer: str) -> Image.Image:
    """
    Haalt de legenda PNG op via LegendURL uit WDM GetCapabilities.
    Returnt een PIL.Image zodat jouw place_legend_on_image() direct werkt.
    """
    cap = requests.get(
        WDM_WMS,
        params={"SERVICE": "WMS", "REQUEST": "GetCapabilities"},
        timeout=60
    )
    cap.raise_for_status()
    root = ET.fromstring(cap.text)

    # zoek Layer <Name> == layer
    for lyr in root.iter():
        if not lyr.tag.endswith("Layer"):
            continue
        name_el = lyr.find("./{*}Name")
        if name_el is None or (name_el.text or "").strip() != layer:
            continue

        online = lyr.find(".//{*}Style/{*}LegendURL/{*}OnlineResource")
        if online is None:
            raise ValueError(f"Geen LegendURL gevonden voor layer='{layer}'")

        href = None
        for k, v in online.attrib.items():
            if k.endswith("href"):   # xlink:href
                href = v
                break
        if not href:
            raise ValueError("LegendURL OnlineResource heeft geen href attribuut")

        r = requests.get(href, timeout=60)
        r.raise_for_status()
        return Image.open(BytesIO(r.content))

    raise ValueError(f"Layer '{layer}' niet gevonden in WDM GetCapabilities")


In [596]:
def download_wdm_png(
    bbox_rd,
    layer=None,
    *,
    layer_name=None,
    out_path="wdm_kaart.png",
    width=2000,
    height=2000,
    add_legend=True
):
    # accepteer zowel layer als layer_name
    if layer is None:
        layer = layer_name
    if not layer:
        raise ValueError("Geef 'layer' of 'layer_name' mee (bijv. bro-grondwaterspiegeldieptemetingen-GHG)")

    params = wms_parameters(bbox_rd, width=width, height=height)
    style = ""

    wdm_img = wms_to_image(WDM_WMS, {
        **params,
        "layers": layer,
        "styles": style,
    })

    if add_legend:
        try:
            legend = wdm_legend_image(layer)  # via LegendURL (GetCapabilities)
            wdm_img = place_legend_on_image(wdm_img, legend, position="bottom-right")
        except Exception as e:
            print(f"[WARN] WDM legenda ophalen mislukt: {e}")

    wdm_img.save(out_path)
    return os.path.abspath(out_path)


In [597]:
PDOK_3D_BASE = "https://api.pdok.nl/kadaster/3d-basisvoorziening/ogc/v1"
EPSG_RD = "EPSG:28992"

In [598]:
RD_CRS = "http://www.opengis.net/def/crs/EPSG/0/28992"

def _bbox_to_str(bbox: Tuple[float, float, float, float]) -> str:
    return ",".join(map(str, bbox))

def _ogc_get_all_features(
    base_url: str,
    collection: str,
    bbox: Tuple[float, float, float, float],
    bbox_crs: str = RD_CRS,
    limit: int = 1000,
    extra_params: Optional[Dict[str, str]] = None,
    timeout: int = 30,
    response_crs: Optional[str] = None,
) -> List[Dict[str, Any]]:
    """
    Haalt ALLE features op voor een collection binnen bbox, incl. pagination (rel=next).
    Werkt met OGC API Features /collections/{collection}/items.

    Belangrijk: veel PDOK OGC API's geven geometrie standaard terug in EPSG:4326.
    Voor DXF wil je RD-coördinaten (EPSG:28992). Daarom proberen we standaard
    `crs=<bbox_crs>` mee te geven (of `response_crs`), en vallen we terug als de server
    die parameter niet accepteert.
    """
    url = f"{base_url.rstrip('/')}/collections/{collection}/items"

    crs_out = response_crs or bbox_crs
    params = {
        "bbox": _bbox_to_str(bbox),
        "bbox-crs": bbox_crs,
        "limit": str(limit),
        "f": "json",
    }

    # Vraag output CRS aan (als ondersteund)
    if crs_out:
        params["crs"] = crs_out

    if extra_params:
        params.update(extra_params)

    features: List[Dict[str, Any]] = []

    while True:
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
        except requests.HTTPError as e:
            # Fallback: sommige endpoints accepteren `crs` niet.
            if params and "crs" in params:
                params = dict(params)
                params.pop("crs", None)
                r = requests.get(url, params=params, timeout=timeout)
                r.raise_for_status()
            else:
                raise e

        data = r.json()
        feats = data.get("features", [])
        features.extend(feats)

        # Pagination: zoek rel="next"
        next_url = None
        for link in data.get("links", []):
            if link.get("rel") == "next" and link.get("href"):
                next_url = link["href"]
                break

        if not next_url:
            break

        # Volgende page: OGC geeft meestal een volledige URL terug.
        url = next_url
        params = None  # next-link bevat vaak al alle params

    return features

def _ensure_layer(doc, name: str, color: int = 7) -> None:
    if name not in doc.layers:
        doc.layers.new(name=name, dxfattribs={"color": color})

def _add_polygon_like_to_dxf(
    msp,
    geom,
    layer: str,
) -> None:
    """
    Voegt Polygon/MultiPolygon toe als LWPolyline(s) op layer.
    """
    if geom.is_empty:
        return

    gtype = geom.geom_type

    def add_polygon(poly):
        # exterior
        ext = list(poly.exterior.coords)
        if len(ext) >= 3:
            msp.add_lwpolyline(ext, close=True, dxfattribs={"layer": layer})
        # interiors (holes)
        for ring in poly.interiors:
            coords = list(ring.coords)
            if len(coords) >= 3:
                msp.add_lwpolyline(coords, close=True, dxfattribs={"layer": layer})

    if gtype == "Polygon":
        add_polygon(geom)
    elif gtype == "MultiPolygon":
        for poly in geom.geoms:
            add_polygon(poly)
    else:
        # Je kunt dit uitbreiden voor LineString/Point indien nodig
        return
    
def add_georef_image_to_doc(
    doc: ezdxf.EzDxf,
    image_path: str,
    bbox_rd,
    layer: str,
    *,
    fade: int = 60,         # 0..100 (hoger = vager)
    contrast: int = 50,     # 0..100
    brightness: int = 50,   # 0..100
) -> None:
    minx, miny, maxx, maxy = bbox_rd
    width_units = float(maxx - minx)
    height_units = float(maxy - miny)

    if layer not in doc.layers:
        doc.layers.new(name=layer)

    # Let op: pad maken we later relatief bij het saven (zie stap 3)
    img_abs = os.path.abspath(image_path)

    with Image.open(img_abs) as im:
        w_px, h_px = im.size

    img_def = doc.add_image_def(filename=img_abs, size_in_pixel=(w_px, h_px))

    img = doc.modelspace().add_image(
        img_def,
        insert=(minx, miny),
        size_in_units=(width_units, height_units),
        rotation=0,
        dxfattribs={"layer": layer},
    )

    # “Leesbaarheid” standaard (AutoCAD ondersteunt dit doorgaans)
    img.dxf.fade = int(fade)
    img.dxf.contrast = int(contrast)
    img.dxf.brightness = int(brightness)


In [599]:
def export_cad_onderlegger_dxf(
    out_path: str,
    bbox_rd: Tuple[float, float, float, float],
    kadas_base_url: str = "https://api.pdok.nl/kadaster/brk-kadastrale-kaart/ogc/v1",
    include_percelen: bool = True,
    include_bebouwing: bool = True,
) -> str:
    """
    Maakt een DXF-onderlegger (RD / EPSG:28992)
    met standaard rasters ONDER de vectorlagen.
    """
    doc = ezdxf.new(setup=True)
    msp = doc.modelspace()

    # ===============================
    # RASTERS (EERST → onderop)
    # ===============================

    # --- LUCHTFOTO (standaard AAN) ---
    if "00_LUCHTFOTO" not in doc.layers:
        doc.layers.new(name="00_LUCHTFOTO")
    layer_lf = doc.layers.get("00_LUCHTFOTO")
    layer_lf.dxf.plot = 1        # plot aan
    try:
        layer_lf.on()
        layer_lf.thaw()
        layer_lf.lock()
    except Exception:
        pass

    add_georef_image_to_doc(
        doc=doc,
        image_path="luchtfoto_percelen.png",
        bbox_rd=bbox_rd,
        layer="00_LUCHTFOTO",
        fade=0,
        contrast=50,
        brightness=50,
    )
        # --- Bestemmingsplankaart---
    if "00_BESTEMMING" not in doc.layers:
        doc.layers.new(name="00_BESTEMMING")
    layer_lf = doc.layers.get("00_BESTEMMING")
    layer_lf.dxf.plot = 1        # plot aan
    try:
        layer_lf.off()
        layer_lf.thaw()
        layer_lf.lock()
    except Exception:
        pass

    add_georef_image_to_doc(
        doc=doc,
        image_path="bestemming_percelen.png",
        bbox_rd=bbox_rd,
        layer="00_BESTEMMING",
        fade=0,
        contrast=50,
        brightness=50,
    )

    # --- AHN (standaard UIT) ---
    if "00_AHN_RASTER" not in doc.layers:
        doc.layers.new(name="00_AHN_RASTER")
    layer_ahn = doc.layers.get("00_AHN_RASTER")
    layer_ahn.dxf.plot = 1       # plot aan (mag ook 0 als je nooit wil plotten)
    try:
        layer_ahn.off()
        layer_ahn.thaw()
        layer_ahn.lock()
    except Exception:
        pass
    

    add_georef_image_to_doc(
        doc=doc,
        image_path="ahn_kaart.png",
        bbox_rd=bbox_rd,
        layer="00_AHN_RASTER",
        fade=0,
        contrast=50,
        brightness=50,
    )


    # --- WDM (standaard UIT) ---
    if "00_WDM_GHG" not in doc.layers:
        doc.layers.new(name="00_WDM_GHG")
    layer_wdm = doc.layers.get("00_WDM_GHG")
    layer_wdm.dxf.plot = 1
    try:
        layer_wdm.off()
        layer_wdm.thaw()
        layer_wdm.lock()
    except Exception:
        pass
    
    add_georef_image_to_doc(
        doc=doc,
        image_path="wdm_ghg.png",
        bbox_rd=bbox_rd,
        layer="00_WDM_GHG",
        fade=0,
        contrast=50,
        brightness=50,
    )
    # ===============================
    # VECTOR-LAYERS
    # ===============================

    if include_percelen:
        _ensure_layer(doc, "01_KAD_PERCELEN", color=3)
    if include_bebouwing:
        _ensure_layer(doc, "02_KAD_BEBouwing", color=1)

    # ===============================
    # VECTOR DATA (komt bovenop rasters)
    # ===============================

    if include_percelen:
        feats = _ogc_get_all_features(
            base_url=kadas_base_url,
            collection="perceel",
            bbox=bbox_rd,
            bbox_crs=RD_CRS,
            limit=1000,
        )
        for f in feats:
            g = f.get("geometry")
            if not g:
                continue
            geom = shape(g)
            _add_polygon_like_to_dxf(msp, geom, layer="01_KAD_PERCELEN")

    if include_bebouwing:
        feats = _ogc_get_all_features(
            base_url=kadas_base_url,
            collection="bebouwing",
            bbox=bbox_rd,
            bbox_crs=RD_CRS,
            limit=1000,
        )
        for f in feats:
            g = f.get("geometry")
            if not g:
                continue
            geom = shape(g)
            _add_polygon_like_to_dxf(msp, geom, layer="02_KAD_BEBouwing")

    # ===============================
    # IMAGE PATHS RELATIEF MAKEN
    # ===============================

    dxf_dir = os.path.dirname(os.path.abspath(out_path))
    for obj in doc.objects:  # compatibel met oudere ezdxf
        if obj.dxftype() == "IMAGEDEF":
            fn = obj.dxf.filename
            if os.path.isabs(fn):
                obj.dxf.filename = os.path.relpath(fn, start=dxf_dir)

    doc.saveas(out_path)
    return out_path


In [600]:
# 1) AHN PNG opslaan (zorg dat hij naast de DXF komt te staan)
ahn_png = download_ahn_png(bbox, out_path="ahn_kaart.png", width=2000, height=2000)
wdm_png = download_wdm_png(
    bbox,
    layer_name="bro-grondwaterspiegeldieptemetingen-GHG",
    out_path="wdm_ghg.png",
    width=2000,
    height=2000
)
dxf_path = export_cad_onderlegger_dxf(
    out_path="onderlegger_snw_met_rasters.dxf",
    bbox_rd=bbox,
    include_percelen=True,
    include_bebouwing=True
)

Tekst